In [16]:
import duckdb, httpx
import os, random, json, time, traceback
from rich import print
import bikidata
import pandas as pd

# Demonstrating non-trival dataset sizes using DBLP

To compare the performance of bikiDATA with a current state-of-the-art triplestore, we index a non-trivial dataset with both and run some queries. We use the DBLP dataset as mentioned in the "Sparqloscope: A Generic Benchmark for the Comprehensive and Concise Performance Evaluation of SPARQL Engines" paper, [downloaded from here](https://drops.dagstuhl.de/entities/artifact/10.4230/dblp.rdf.ntriples.2025-04-01).

In [ ]:
!wget https://drops.dagstuhl.de/storage/artifacts/dblp/rdf/2025/dblp-2025-04-01.nt.gz

In [ ]:
bikidata.build(["dblp-2025-04-01.nt.gz"])

DEBUG     bikidata 2026-03-09 17:56:38 Building Bikidata index with ['dblp-2025-04-01.nt.gz']
DEBUG     bikidata 2026-03-09 17:56:38 Good, there are no triples in bikidate table yet


We perform fulltext queries, or select some random items to view.

In [27]:
r = bikidata.query({'filters':[{'p':'fts', 'o':'Knuth'}], 'size':4})
print(r['total'], "in total")
for x in r['results'].values():
    print(x)

877 in total

{
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#AuthorSignature>',
        '<https://dblp.org/rdf/schema#Signature>'
    ],
    '<https://dblp.org/rdf/schema#signatureCreator>': ['<https://dblp.org/pid/k/DonaldEKnuth>'],
    '<https://dblp.org/rdf/schema#signaturePublication>': ['<https://dblp.org/rec/books/daglib/0000774>'],
    '<https://dblp.org/rdf/schema#signatureDblpName>': ['"Donald E. Knuth"'],
    '<https://dblp.org/rdf/schema#signatureOrdinal>': ['"1"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    'id': '_:Sig_2254194596d5922a7bc4286d10fd6d2b_1',
    'graph': []
}

{
    '<https://dblp.org/rdf/schema#signatureOrdinal>': ['"1"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    '<https://dblp.org/rdf/schema#signaturePublication>': ['<https://dblp.org/rec/journals/corr/abs-2005-05421>'],
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#AuthorSignature>',
        '<https://dblp.org/rdf/schema#Signature>'
    ],
    '<https://dblp.org/rdf/schema#signatureDblpName>': ['"Craig Knuth"'],
    '<https://dblp.org/rdf/schema#signatureCreator>': ['<https://dblp.org/pid/264/9782>'],
    'id': '_:Sig_40d72ef4efef14d4a35b274f34baadd5_1',
    'graph': []
}

{
    '<https://dblp.org/rdf/schema#signatureOrdinal>': ['"1"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    '<https://dblp.org/rdf/schema#signatureCreator>': ['<https://dblp.org/pid/57/8029>'],
    '<https://dblp.org/rdf/schema#signaturePublication>': ['<https://dblp.org/rec/conf/otm/KnuthHS16>'],
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#AuthorSignature>',
        '<https://dblp.org/rdf/schema#Signature>'
    ],
    '<https://dblp.org/rdf/schema#signatureDblpName>': ['"Magnus Knuth"'],
    'id': '_:Sig_5e204a1e7ba56dc9abca0d4caf5cf94a_1',
    'graph': []
}

{
    '<http://purl.org/spar/datacite/hasIdentifier>': ['_:ID_61bf66ed46f7c0685f8b4b111a117e69'],
    '<https://dblp.org/rdf/schema#primaryCreatorName>': ['"Eric J. Knuth"'],
    '<https://dblp.org/rdf/schema#creatorName>': ['"Eric J. Knuth"'],
    '<http://www.w3.org/2000/01/rdf-schema#label>': ['"Eric J. Knuth"'],
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#Creator>',
        '<https://dblp.org/rdf/schema#Person>'
    ],
    'id': '<https://dblp.org/pid/220/7978>',
    'graph': []
}

In [30]:
r = bikidata.query({'filters':[{'p':'id', 'o':'random 3'}]})
for x in r['results'].values():
    print(x)

{
    '<http://purl.org/spar/datacite/hasIdentifier>': [
        '_:ID_e65e9446514fc299784f81f1ffb82d1a',
        '_:ID_cdcea05162dfd196dbfb72ae449cd37d'
    ],
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#Reference>',
        '<https://dblp.org/rdf/schema#Publication>'
    ],
    '<http://www.w3.org/2002/07/owl#sameAs>': [
        '<https://doi.org/10.1007/978-0-387-31439-6_100227>',
        '<http://dx.doi.org/10.1007/978-0-387-31439-6_100227>'
    ],
    '<https://dblp.org/rdf/schema#pagination>': ['"691"'],
    '<https://dblp.org/rdf/schema#doi>': ['<https://doi.org/10.1007/978-0-387-31439-6_100227>'],
    '<https://dblp.org/rdf/schema#documentPage>': ['<https://doi.org/10.1007/978-0-387-31439-6_100227>'],
    '<https://dblp.org/rdf/schema#listedOnTocPage>': ['<https://dblp.org/db/reference/vision/vision2014>'],
    '<https://dblp.org/rdf/schema#title>': ['"Robust Clustering."'],
    '<https://dblp.org/rdf/schema#bibtexType>': ['<http://purl.org/net/nknouf/ns/bibtex#Incollection>'],
    '<https://dblp.org/rdf/schema#publishedIn>': ['"Computer Vision, A Reference Guide"'],
    '<https://dblp.org/rdf/schema#yearOfPublication>': ['"2014"^^<http://www.w3.org/2001/XMLSchema#gYear>'],
    '<https://dblp.org/rdf/schema#numberOfCreators>': ['"0"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    '<https://dblp.org/rdf/schema#publishedInBook>': ['"Computer Vision, A Reference Guide"'],
    '<https://dblp.org/rdf/schema#primaryDocumentPage>': ['<https://doi.org/10.1007/978-0-387-31439-6_100227>'],
    '<http://www.w3.org/2000/01/rdf-schema#label>': ['"Robust Clustering. (2014)"'],
    'id': '<https://dblp.org/rec/reference/vision/X14hh>',
    'graph': []
}

{
    '<https://dblp.org/rdf/schema#documentPage>': ['<https://doi.org/10.1145/317825.317876>'],
    '<https://dblp.org/rdf/schema#hasSignature>': ['_:Sig_91ab64695839d2f122d0927ba7714ef2_1'],
    '<http://www.w3.org/2002/07/owl#sameAs>': [
        '<http://dx.doi.org/10.1145/317825.317876>',
        '<https://doi.org/10.1145/317825.317876>'
    ],
    '<http://purl.org/spar/datacite/hasIdentifier>': [
        '_:ID_21dc5ec12519862a59fb2b96824f6677',
        '_:ID_24fc6b7ec9f8ebadb6219991193067ae'
    ],
    '<https://dblp.org/rdf/schema#bibtexType>': ['<http://purl.org/net/nknouf/ns/bibtex#Inproceedings>'],
    '<https://dblp.org/rdf/schema#authoredBy>': ['<https://dblp.org/pid/60/2881>'],
    '<https://dblp.org/rdf/schema#publishedIn>': ['"DAC"'],
    '<https://dblp.org/rdf/schema#pagination>': ['"312-318"'],
    '<https://dblp.org/rdf/schema#yearOfEvent>': ['"1985"^^<http://www.w3.org/2001/XMLSchema#gYear>'],
    '<http://www.w3.org/2000/01/rdf-schema#label>': [
        '"Thomas R. Smith: A data architecture for an uncertain design and manufacturing environment. (1985)"'
    ],
    '<https://dblp.org/rdf/schema#createdBy>': ['<https://dblp.org/pid/60/2881>'],
    '<https://dblp.org/rdf/schema#title>': [
        '"A data architecture for an uncertain design and manufacturing environment."'
    ],
    '<https://dblp.org/rdf/schema#numberOfCreators>': ['"1"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    '<https://dblp.org/rdf/schema#primaryDocumentPage>': ['<https://doi.org/10.1145/317825.317876>'],
    '<https://dblp.org/rdf/schema#publishedAsPartOf>': ['<https://dblp.org/rec/conf/dac/1985>'],
    '<https://dblp.org/rdf/schema#publishedInStream>': ['<https://dblp.org/streams/conf/dac>'],
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#Inproceedings>',
        '<https://dblp.org/rdf/schema#Publication>'
    ],
    '<https://dblp.org/rdf/schema#doi>': ['<https://doi.org/10.1145/317825.317876>'],
    '<https://dblp.org/rdf/schema#listedOnTocPage>': ['<https://dblp.org/db/conf/dac/dac1985>'],
    '<https://dblp.org/rdf/schema#publishedInBook>': ['"DAC"'],
    '<https://dblp.org/rdf/schema#yearOfPublication>': ['"1985"^^<http://www.w3.org/2001/XMLSchema#gYear>'],
    'id': '<https://dblp.org/rec/conf/dac/Smith85>',
    'graph': []
}

{
    '<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>': [
        '<https://dblp.org/rdf/schema#AuthorSignature>',
        '<https://dblp.org/rdf/schema#Signature>'
    ],
    '<https://dblp.org/rdf/schema#signatureDblpName>': ['"Richard Berlin"'],
    '<https://dblp.org/rdf/schema#signatureCreator>': ['<https://dblp.org/pid/83/6265>'],
    '<https://dblp.org/rdf/schema#signaturePublication>': ['<https://dblp.org/rec/conf/dac/WinslettBPW85>'],
    '<https://dblp.org/rdf/schema#signatureOrdinal>': ['"2"^^<http://www.w3.org/2001/XMLSchema#integer>'],
    'id': '_:Sig_c8de18318571c9675acf05dd33da33e8_2',
    'graph': []
}

# Comparison with Sparqloscope queries

To do an evaluation of the performance, we compare some SPARQL queries from the Sparqloscope paper. We selected queries that had a runtime of more than a few seconds in the published results. The same query was then converted into SQL that can be run on the bikidata engine. A Qlever triplestore database was run on the same machine on which the bikidata DuckDB queries were run.

In [34]:
queries = {}
for filename in os.listdir('.'):
    if not (filename.endswith('.sql') or filename.endswith('.sparql')):
        continue
    base, ext = filename.split('.')
    contents = open(filename).read()
    obj = queries.setdefault(base, {})
    obj[ext] = contents
print(len(queries))

14

In [35]:
for i, q in queries.items():
    print(f'[bold green]{i} {'#'*80} [/bold green]')
    print(q['sparql'])
    print('-'*80)
    print(q['sql'])

strbefore ################################################################################ 

PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 

SELECT (SUM(STRLEN(STRBEFORE(?o, "a"))) AS ?checksum) { ?s rdfs:label ?o }

--------------------------------------------------------------------------------

SELECT SUM(LENGTH(
  CASE WHEN STRPOS(lit.value, 'a') > 0
       THEN LEFT(lit.value, STRPOS(lit.value, 'a') - 2)
       ELSE ''
  END
)) AS checksum
FROM triples AS t1
JOIN iris AS i1 ON i1.hash = t1.p AND i1.value = '<http://www.w3.org/2000/01/rdf-schema#label>'
JOIN literals AS lit ON lit.hash = t1.o

strlen ################################################################################ 

...

SELECT COUNT(*) AS count
FROM triples AS t1
JOIN iris AS i1 ON i1.hash = t1.p AND i1.value = '<http://www.w3.org/2000/01/rdf-schema#label>'
JOIN literals AS lit ON lit.hash = t1.o
WHERE regexp_matches(lit.value, 'com')

distinct-count-object-low-multipl ################################################################################ 

PREFIX dblp: <https://dblp.org/rdf/schema#>

SELECT (COUNT(DISTINCT ?object) AS ?count)
WHERE {
  ?subject dblp:hasSignature ?object .
}

--------------------------------------------------------------------------------

SELECT COUNT(DISTINCT t1.o) AS count
FROM triples AS t1
JOIN iris AS i1 ON i1.hash = t1.p AND i1.value = '<https://dblp.org/rdf/schema#hasSignature>'

In [42]:
endpoint = "http://localhost:60600/" # In this case we were running QLever on port 60600
for i, query in queries.items():
    print(i)
    try:
        start = time.time()        
        response = httpx.post(
          endpoint,
          timeout=360,
          data={"query": query['sparql']},
          headers={"Accept": "application/sparql-results+json"}
        )
        end = time.time()
        query['sparql_result_raw'] = response.json() 
        query['sparql_duration'] = end-start
    except Exception as e:
        query['sparql_error'] = str(e)
print("Done!")    

strbefore

strlen

transitive-path-large-join

strafter

number-of-blank-nodes

...

strends

group-by-string-groupconcat

regex-3-fixed

distinct-count-object-low-multipl

Done!

In [38]:
for q in queries.values():
    qq = q['sparql_result_raw'].get('results', {}).get('bindings', [None])[0]
    if qq:    
        q['sparql_result'] = list(qq.values())[0]['value']
    else:
        q['sparql_result'] = None
    # print(q['sparql_result_raw'].get('results', {}).get('bindings', ["??"])[0])


In [ ]:
for i, query in queries.items():
    print(i)
    try:
        if 'sql_error' in query:
            del query['sql_error']
        start = time.time()
        result = bikidata.raw().execute(query['sql']).fetchall()
        query['sql_result'] = result[0][0]
        end = time.time()
        query['sql_duration'] = end-start
    except Exception as e:
        traceback.print_exc()
        query['sql_error'] = str(e)
print("Done!")

# Cold start QLever, running the queries first time

In [41]:
exclude = {"sparql", "sql", "sparql_result_raw"}
df = pd.DataFrame([{k: v for k, v in d.items() if k not in exclude} for d in queries.values()])
df

,sql_result,sql_duration,sparql_duration,sparql_result,name
0,68610010,1.761238,18.971536,68607791,strbefore
1,939793994,1.734807,15.265711,916588643,strlen
2,0,1.620833,0.938576,0,transitive-path-large-join
3,838049269,1.817127,19.687145,827145216,strafter
4,503926494,1.451660,5.543682,241373570,number-of-blank-nodes
5,729713,1.799484,15.854430,729715,regex-3
6,466960,1.725058,14.982124,466960,regex-3-contains
7,26034578,15.243499,11.796228,26034578,exists-join-3-chain-2
8,40,1.777630,14.552509,40,strstarts
9,52069156,2.484082,7.627791,52069156,exists-join-3-chain-1


We see that the bikiData query execution is consistently significantly faster than the equivalent SPARQL queries in all cases. If we however re-run the queries again, the performance radically improves as QLever caches have warmed up.

# Re-run SPARQL queries with warm cache

In [44]:
# re-run the sparql query cell above and re-plot the results
exclude = {"sparql", "sql", "sparql_result_raw"}
df = pd.DataFrame([{k: v for k, v in d.items() if k not in exclude} for d in queries.values()])
df

,sql_result,sql_duration,sparql_duration,sparql_result,name
0,68610010,1.761238,0.030986,68607791,strbefore
1,939793994,1.734807,0.027595,916588643,strlen
2,0,1.620833,1.084875,0,transitive-path-large-join
3,838049269,1.817127,0.026809,827145216,strafter
4,503926494,1.451660,0.027290,241373570,number-of-blank-nodes
5,729713,1.799484,0.026106,729715,regex-3
6,466960,1.725058,0.044399,466960,regex-3-contains
7,26034578,15.243499,0.031203,26034578,exists-join-3-chain-2
8,40,1.777630,0.027216,40,strstarts
9,52069156,2.484082,0.028053,52069156,exists-join-3-chain-1
